# 04 — Model Training

Train 4 models using chronological (time-based) data splits.

| Model | Type |
|-------|------|
| Historical Baseline | Naive (previous year) |
| Linear Regression | Linear |
| Random Forest | Ensemble (200 trees) |
| XGBoost | Gradient Boosting |


In [1]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import plotly.express as px
from config import *
from src.data_loader import load_all_datasets
from src.preprocessing import (
    preprocess_subnational_tree_loss, preprocess_carbon_data, merge_carbon_with_loss
)
from src.feature_engineering import build_feature_set, get_feature_columns
from src.train import chronological_split, train_all_models


## Prepare Data


In [2]:
datasets = load_all_datasets()
processed = preprocess_subnational_tree_loss(
    datasets['subnational_loss'], threshold=30, country='Brazil'
)
carbon = preprocess_carbon_data(
    datasets['subnational_carbon'], threshold=30, country='Brazil'
)
processed = merge_carbon_with_loss(processed, carbon)
featured = build_feature_set(processed)
feature_cols = get_feature_columns(featured)
print(f'Features: {len(feature_cols)}')
print(f'Total rows: {len(featured)}')


2026-11-21 00:15:18,335 - deforestation_ml.data_loader - INFO - Loaded Country Tree Cover Loss: 1888 rows × 28 columns from Country tree cover loss.csv


2026-11-21 00:15:18,358 - deforestation_ml.data_loader - INFO - Loaded Subnational Tree Cover Loss: 28000 rows × 29 columns from Subnational 1 tree cover loss.csv


2026-11-21 00:15:18,361 - deforestation_ml.data_loader - INFO - Loaded Country Carbon Data: 1888 rows × 30 columns from Country carbon data.csv


2026-11-21 00:15:18,383 - deforestation_ml.data_loader - INFO - Loaded Subnational Carbon Data: 28000 rows × 31 columns from Subnational 1 carbon data.csv


2026-11-21 00:15:18,387 - deforestation_ml.preprocessing - INFO - Filtered to threshold=30: 3500 rows (from 28000)


2026-11-21 00:15:18,389 - deforestation_ml.preprocessing - INFO - Filtered to country='Brazil': 27 rows


... (4 outputs left out)

2026-11-21 00:15:18,427 - deforestation_ml.feature_engineering - INFO - Starting feature engineering...


2026-11-21 00:15:18,497 - deforestation_ml.feature_engineering - INFO - Encoded 27 regions


2026-11-21 00:15:18,498 - deforestation_ml.feature_engineering - INFO - Feature engineering complete: 15 features created


Features: 15
Total rows: 594


## Chronological Train/Val/Test Split

- **Train:** years ≤ 2017
- **Validation:** 2018–2019
- **Test:** 2020–2022

No random shuffling — prevents temporal information leakage.


In [3]:
splits = chronological_split(
    featured,
    train_end_year=TRAIN_END_YEAR,
    val_end_year=VAL_END_YEAR,
    feature_cols=feature_cols,
)
print(f'Train: {len(splits["train"][0])} rows (years ≤ {TRAIN_END_YEAR})')
print(f'Val:   {len(splits["val"][0])} rows ({TRAIN_END_YEAR+1}–{VAL_END_YEAR})')
print(f'Test:  {len(splits["test"][0])} rows ({VAL_END_YEAR+1}–{YEAR_END})')


2026-11-21 00:15:18,512 - deforestation_ml.train - INFO - Chronological split — Train: 351 rows (≤2017), Val: 54 rows (2018–2019), Test: 81 rows (>2019)


Train: 351 rows (years ≤ 2017)
Val:   54 rows (2018–2019)
Test:  81 rows (2020–2022)


## Train All Models


In [4]:
trained_models = train_all_models(splits)
print(f'\nModels trained: {list(trained_models.keys())}')


2026-11-21 00:15:18,521 - deforestation_ml.train - INFO - Training Model A: Historical Baseline...


2026-11-21 00:15:18,522 - deforestation_ml.train - INFO - Training Model B: Linear Regression...


2026-11-21 00:15:18,563 - deforestation_ml.train - INFO - Linear Regression trained. R² (train): 0.7903


2026-11-21 00:15:18,564 - deforestation_ml.train - INFO - Training Model C: Random Forest...


2026-11-21 00:15:18,811 - deforestation_ml.train - INFO - Random Forest trained. R² (train): 0.8641


2026-11-21 00:15:18,811 - deforestation_ml.train - INFO - Training Model D: XGBoost...


... (2 outputs left out)

2026-11-21 00:15:19,254 - deforestation_ml.train - INFO - Saved Linear Regression to D:\Project\Rainforest Deforestation Monitoring and Prediction Using Machine Learning\models\linear_regression_bundle.joblib


2026-11-21 00:15:19,314 - deforestation_ml.train - INFO - Saved Random Forest to D:\Project\Rainforest Deforestation Monitoring and Prediction Using Machine Learning\models\random_forest_bundle.joblib


2026-11-21 00:15:19,320 - deforestation_ml.train - INFO - Saved XGBoost to D:\Project\Rainforest Deforestation Monitoring and Prediction Using Machine Learning\models\xgboost_bundle.joblib



Models trained: ['Historical Baseline', 'Linear Regression', 'Random Forest', 'XGBoost']


## Feature Importance — Random Forest


In [5]:
rf_model = trained_models['Random Forest']['model']
fi = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True).tail(15)

fig = px.bar(
    fi, x='Importance', y='Feature', orientation='h',
    title='Random Forest — Top 15 Feature Importances',
    template='plotly_white'
)
fig.show()


## Feature Importance — XGBoost


In [6]:
if 'XGBoost' in trained_models:
    xgb_model = trained_models['XGBoost']['model']
    fi_xgb = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': xgb_model.feature_importances_
    }).sort_values('Importance', ascending=True).tail(15)
    fig = px.bar(
        fi_xgb, x='Importance', y='Feature', orientation='h',
        title='XGBoost — Top 15 Feature Importances',
        template='plotly_white'
    )
    fig.show()
else:
    print('XGBoost not available.')


## Linear Regression Coefficients


In [7]:
lr_model = trained_models['Linear Regression']['model']
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': lr_model.coef_
}).sort_values('Coefficient', key=abs, ascending=False)
print('Linear Regression Coefficients (standardized):')
print(coef_df.to_string(index=False))


Linear Regression Coefficients (standardized):
                   Feature   Coefficient
  tc_loss_ha_roll_mean_5yr 149912.172868
   tc_loss_ha_roll_std_5yr -54435.552344
            loss_trend_3yr  46181.944124
   tc_loss_ha_roll_std_3yr  42112.121870
        loss_rate_pct_lag1 -23578.153807
  cumulative_loss_pct_lag1  19882.807266
         loss_acceleration -18278.546239
           tc_loss_ha_lag1  16129.885867
               loss_cv_5yr -12721.003033
           tc_loss_ha_lag2  11477.335135
    loss_change_1yr_lagged  10862.781142
           tc_loss_ha_lag3  -7833.088202
  tc_loss_ha_roll_mean_3yr   6835.512964
            region_encoded  -1802.068553
loss_change_pct_1yr_lagged    464.857858


## Conclusion

- 4 models trained on 351 training observations (≤2017)
- `tc_loss_ha_lag1` is consistently the most important feature
- Random Forest and XGBoost capture nonlinear patterns
- Model artifacts saved to `models/` directory

**Note:** Feature importance shows statistical associations, not causation.
